In [ ]:
# Week 4: Product Performance Analysis

import os
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

data_path = '../data/'
sales_df = pd.read_csv(os.path.join(data_path, 'engineered_sales.csv'))
products_df = pd.read_csv(os.path.join(data_path, 'validated_products.csv'))
inventory_df = pd.read_csv(os.path.join(data_path, 'engineered_inventory.csv'))

sales_df['order_date'] = pd.to_datetime(sales_df['order_date'])

# Aggregate KPIs
product_kpis = sales_df.groupby('product_id').agg(
    total_units_sold=('quantity', 'sum'),
    total_revenue=('total_revenue', 'sum'),
    total_gross_profit=('gross_profit', 'sum'),
    number_of_orders=('order_id', 'nunique'),
    avg_selling_price=('unit_price', 'mean')
).reset_index()

product_perf = products_df.merge(product_kpis, on='product_id', how='left').fillna({
    'total_units_sold': 0,
    'total_revenue': 0,
    'total_gross_profit': 0,
    'number_of_orders': 0
})

# Fast vs Slow Classification
q75_units = product_perf['total_units_sold'].quantile(0.75)
q25_units = product_perf['total_units_sold'].quantile(0.25)

def classify_mover(row):
    if row['total_units_sold'] == 0:
        return 'No Sales'
    elif row['total_units_sold'] >= q75_units:
        return 'Fast-Moving'
    elif row['total_units_sold'] <= q25_units:
        return 'Slow-Moving'
    else:
        return 'Moderate'

product_perf['movement_classification'] = product_perf.apply(classify_mover, axis=1)

output_path = '../data/processed/'
os.makedirs(output_path, exist_ok=True)
product_perf.to_csv(os.path.join(output_path, 'product_performance_summary.csv'), index=False)
